# 歪-励磁磁場曲線に関する調査  
作成日：2026年09月02日  
作成者：上杉 健太  

In [ ]:
# conf
project_name = "260902_歪-励磁磁場曲線"
master_csv_path = './MasterData/FolderMaster.csv'

## 計測波形の表示

In [ ]:
import glob
from pathlib import Path
import pandas as pd

# 1. FolderMaster.csv の読み込み
df_master = pd.read_csv(master_csv_path)

# WithCancel の表記をブール値 (True/False) に統一
if df_master['WithCancel'].dtype == object:
  df_master['WithCancel'] = (
      df_master['WithCancel']
      .astype(str)
      .str.strip()
      .str.lower()
      .map({'true': True, 'false': False, '1': True, '0': False})
  )

# フォルダ名 -> WithCancel の対応辞書を作成
cancel_map = dict(zip(df_master['FolderName'], df_master['WithCancel']))

# 2. 対象となるCSVファイルの取得とグループ分け
target_pattern = f'source/{project_name}/**/*.csv'
all_csv_files = sorted(glob.glob(target_pattern, recursive=True))

csv_files_false = []
csv_files_true = []

for file_path in all_csv_files:
  # CSVが格納されているフォルダ名を取得 (例: source/26.../700turns0.5A/data.csv -> 700turns0.5A)
  folder_name = Path(file_path).parts[-2]
  with_cancel_status = cancel_map.get(folder_name, None)

  if with_cancel_status is False:
    csv_files_false.append(file_path)
  elif with_cancel_status is True:
    csv_files_true.append(file_path)

print(
    f'分類結果: WithCancel=False: {len(csv_files_false)}件 / WithCancel=True: {len(csv_files_true)}件'
)

In [ ]:
import matplotlib.pyplot as plt

# 日本語フォントの設定
try:
  import japanize_matplotlib
except ImportError:
  plt.rcParams['font.family'] = 'sans-serif'

# --- WithCancel = False の波形縦並びプロット ---
if not csv_files_false:
  print('WithCancel = False の対象CSVファイルが見つかりませんでした。')
else:
  num_files = len(csv_files_false)

  fig, axes = plt.subplots(
      nrows=num_files, ncols=1, figsize=(10, 2.5 * num_files), sharex=True
  )

  if num_files == 1:
    axes = [axes]

  x_col = '時間[μs]'
  y_col = ' 加算平均値[V]'

  for ax, file_path in zip(axes, csv_files_false):
    try:
      df = pd.read_csv(file_path, encoding='ms932')
    except (UnicodeDecodeError, pd.errors.ParserError):
      df = pd.read_csv(file_path, encoding='cp932')

    if x_col in df.columns and y_col in df.columns:
      folder_and_file = '/'.join(Path(file_path).parts[-2:])
      y_data_mv = df[y_col] * 1000  # V -> mV

      ax.plot(
          df[x_col],
          y_data_mv,
          color='tab:blue',
          linewidth=1.2,
          label=folder_and_file,
      )

      ax.set_title(
          f'{folder_and_file} (WithCancel = False)',
          fontsize=10,
          loc='left',
          fontweight='bold',
      )
      ax.set_ylabel('加算平均値[mV]', fontsize=9)
      ax.set_xlim(0, 700)
      ax.set_ylim(-5, 5)
      ax.grid(True, linestyle='--', alpha=0.6)
    else:
      ax.text(
          0.5,
          0.5,
          f'指定カラムが見つかりません:\n{file_path}',
          ha='center',
          va='center',
          transform=ax.transAxes,
      )

  axes[-1].set_xlabel(x_col, fontsize=10)

  # 保存処理
  output_dir = Path(f'./results/{project_name}')
  output_dir.mkdir(parents=True, exist_ok=True)
  output_filename = output_dir / 'comparison_plot_WithCancel_False.png'

  plt.tight_layout()
  plt.savefig(output_filename, dpi=300, bbox_inches='tight')
  plt.show()

  print(
      f'WithCancel = False: 計 {num_files} 件のグラフを作成し、"{output_filename}" に保存しました。'
  )

In [ ]:
import matplotlib.pyplot as plt

# 日本語フォントの設定
try:
  import japanize_matplotlib
except ImportError:
  plt.rcParams['font.family'] = 'sans-serif'

# --- WithCancel = True の波形縦並びプロット ---
if not csv_files_true:
  print('WithCancel = True の対象CSVファイルが見つかりませんでした。')
else:
  num_files = len(csv_files_true)

  fig, axes = plt.subplots(
      nrows=num_files, ncols=1, figsize=(10, 2.5 * num_files), sharex=True
  )

  if num_files == 1:
    axes = [axes]

  x_col = '時間[μs]'
  y_col = ' 加算平均値[V]'

  for ax, file_path in zip(axes, csv_files_true):
    try:
      df = pd.read_csv(file_path, encoding='ms932')
    except (UnicodeDecodeError, pd.errors.ParserError):
      df = pd.read_csv(file_path, encoding='cp932')

    if x_col in df.columns and y_col in df.columns:
      folder_and_file = '/'.join(Path(file_path).parts[-2:])
      y_data_mv = df[y_col] * 1000  # V -> mV

      ax.plot(
          df[x_col],
          y_data_mv,
          color='tab:orange',
          linewidth=1.2,
          label=folder_and_file,
      )

      ax.set_title(
          f'{folder_and_file} (WithCancel = True)',
          fontsize=10,
          loc='left',
          fontweight='bold',
      )
      ax.set_ylabel('加算平均値[mV]', fontsize=9)
      ax.set_xlim(0, 700)
      ax.set_ylim(-5, 5)
      ax.grid(True, linestyle='--', alpha=0.6)
    else:
      ax.text(
          0.5,
          0.5,
          f'指定カラムが見つかりません:\n{file_path}',
          ha='center',
          va='center',
          transform=ax.transAxes,
      )

  axes[-1].set_xlabel(x_col, fontsize=10)

  # 保存処理
  output_dir = Path(f'./results/{project_name}')
  output_dir.mkdir(parents=True, exist_ok=True)
  output_filename = output_dir / 'comparison_plot_WithCancel_True.png'

  plt.tight_layout()
  plt.savefig(output_filename, dpi=300, bbox_inches='tight')
  plt.show()

  print(
      f'WithCancel = True: 計 {num_files} 件のグラフを作成し、"{output_filename}" に保存しました。'
  )

## ハイパスフィルタを適用した計測波形の表示

In [ ]:
import glob
from pathlib import Path
import pandas as pd

# 1. FolderMaster.csv の読み込み
df_master = pd.read_csv(master_csv_path)

# WithCancel の表記をブール値 (True/False) に標準化
if df_master['WithCancel'].dtype == object:
  df_master['WithCancel'] = (
      df_master['WithCancel']
      .astype(str)
      .str.strip()
      .str.lower()
      .map({'true': True, 'false': False, '1': True, '0': False})
  )

cancel_map = dict(zip(df_master['FolderName'], df_master['WithCancel']))

# 2. 対象となるCSVファイルの取得とグループ分け
target_pattern = f'source/{project_name}/**/*.csv'
csv_files = sorted(glob.glob(target_pattern, recursive=True))

csv_files_false = []
csv_files_true = []

for file_path in csv_files:
  folder_name = Path(file_path).parts[-2]
  status = cancel_map.get(folder_name, None)

  if status is False:
    csv_files_false.append(file_path)
  elif status is True:
    csv_files_true.append(file_path)

print(
    f'分類完了: WithCancel=False: {len(csv_files_false)}件 / WithCancel=True: {len(csv_files_true)}件'
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

# 日本語フォントの設定
try:
  import japanize_matplotlib
except ImportError:
  plt.rcParams['font.family'] = 'sans-serif'

# 1. ハイパスフィルタの設定
CUTOFF_HZ = 200e3  # 200 kHz
FILTER_ORDER = 4

if not csv_files_false:
  print('WithCancel = False の対象CSVファイルが見つかりませんでした。')
else:
  num_files = len(csv_files_false)

  fig, axes = plt.subplots(
      nrows=num_files, ncols=1, figsize=(10, 2.5 * num_files), sharex=True
  )

  if num_files == 1:
    axes = [axes]

  x_col = '時間[μs]'
  y_col = ' 加算平均値[V]'

  for ax, file_path in zip(axes, csv_files_false):
    try:
      df = pd.read_csv(file_path, encoding='ms932')
    except (UnicodeDecodeError, pd.errors.ParserError):
      df = pd.read_csv(file_path, encoding='cp932')

    if x_col in df.columns and y_col in df.columns:
      folder_and_file = '/'.join(Path(file_path).parts[-2:])

      time_us = df[x_col].values
      y_v = df[y_col].values

      # サンプリング周波数(fs)算出
      dt_us = np.mean(np.diff(time_us))
      fs = 1e6 / dt_us
      nyq = 0.5 * fs

      if CUTOFF_HZ >= nyq:
        ax.text(
            0.5,
            0.5,
            f'サンプリング周波数が不足しています\n(fs={fs/1e3:.1f}kHz < 400kHz)',
            ha='center',
            va='center',
            transform=ax.transAxes,
            color='red',
        )
        continue

      # 200kHz HPF適用
      normal_cutoff = CUTOFF_HZ / nyq
      b, a = butter(FILTER_ORDER, normal_cutoff, btype='highpass')

      y_mv = y_v * 1000
      y_filtered = filtfilt(b, a, y_mv)

      ax.plot(
          time_us,
          y_filtered,
          color='tab:blue',
          linewidth=1.2,
          label=folder_and_file,
      )

      ax.set_title(
          f'{folder_and_file} (HPF 200kHz / WithCancel = False)',
          fontsize=10,
          loc='left',
          fontweight='bold',
      )
      ax.set_ylabel('加算平均値[mV]', fontsize=9)
      ax.set_xlim(0, 700)
      ax.set_ylim(-5, 5)
      ax.grid(True, linestyle='--', alpha=0.6)
    else:
      ax.text(
          0.5,
          0.5,
          f'指定カラムが見つかりません:\n{file_path}',
          ha='center',
          va='center',
          transform=ax.transAxes,
      )

  axes[-1].set_xlabel(x_col, fontsize=10)

  # 保存処理
  output_dir = Path(f'./results/{project_name}')
  output_dir.mkdir(parents=True, exist_ok=True)
  output_filename = (
      output_dir / 'comparison_plot_hpf200kHz_WithCancel_False.png'
  )

  plt.tight_layout()
  plt.savefig(output_filename, dpi=300, bbox_inches='tight')
  plt.show()

  print(
      f'WithCancel = False: 計 {num_files} 件のHPF適用グラフを作成し、"{output_filename}" に保存しました。'
  )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

# 日本語フォントの設定
try:
  import japanize_matplotlib
except ImportError:
  plt.rcParams['font.family'] = 'sans-serif'

# 1. ハイパスフィルタの設定
CUTOFF_HZ = 200e3  # 200 kHz
FILTER_ORDER = 4

if not csv_files_true:
  print('WithCancel = True の対象CSVファイルが見つかりませんでした。')
else:
  num_files = len(csv_files_true)

  fig, axes = plt.subplots(
      nrows=num_files, ncols=1, figsize=(10, 2.5 * num_files), sharex=True
  )

  if num_files == 1:
    axes = [axes]

  x_col = '時間[μs]'
  y_col = ' 加算平均値[V]'

  for ax, file_path in zip(axes, csv_files_true):
    try:
      df = pd.read_csv(file_path, encoding='ms932')
    except (UnicodeDecodeError, pd.errors.ParserError):
      df = pd.read_csv(file_path, encoding='cp932')

    if x_col in df.columns and y_col in df.columns:
      folder_and_file = '/'.join(Path(file_path).parts[-2:])

      time_us = df[x_col].values
      y_v = df[y_col].values

      # サンプリング周波数(fs)算出
      dt_us = np.mean(np.diff(time_us))
      fs = 1e6 / dt_us
      nyq = 0.5 * fs

      if CUTOFF_HZ >= nyq:
        ax.text(
            0.5,
            0.5,
            f'サンプリング周波数が不足しています\n(fs={fs/1e3:.1f}kHz < 400kHz)',
            ha='center',
            va='center',
            transform=ax.transAxes,
            color='red',
        )
        continue

      # 200kHz HPF適用
      normal_cutoff = CUTOFF_HZ / nyq
      b, a = butter(FILTER_ORDER, normal_cutoff, btype='highpass')

      y_mv = y_v * 1000
      y_filtered = filtfilt(b, a, y_mv)

      ax.plot(
          time_us,
          y_filtered,
          color='tab:orange',
          linewidth=1.2,
          label=folder_and_file,
      )

      ax.set_title(
          f'{folder_and_file} (HPF 200kHz / WithCancel = True)',
          fontsize=10,
          loc='left',
          fontweight='bold',
      )
      ax.set_ylabel('加算平均値[mV]', fontsize=9)
      ax.set_xlim(0, 700)
      ax.set_ylim(-5, 5)
      ax.grid(True, linestyle='--', alpha=0.6)
    else:
      ax.text(
          0.5,
          0.5,
          f'指定カラムが見つかりません:\n{file_path}',
          ha='center',
          va='center',
          transform=ax.transAxes,
      )

  axes[-1].set_xlabel(x_col, fontsize=10)

  # 保存処理
  output_dir = Path(f'./results/{project_name}')
  output_dir.mkdir(parents=True, exist_ok=True)
  output_filename = (
      output_dir / 'comparison_plot_hpf200kHz_WithCancel_True.png'
  )

  plt.tight_layout()
  plt.savefig(output_filename, dpi=300, bbox_inches='tight')
  plt.show()

  print(
      f'WithCancel = True: 計 {num_files} 件のHPF適用グラフを作成し、"{output_filename}" に保存しました。'
  )

## 各区間内の最小・最大値を見つけてVppを算出する

In [ ]:
import glob
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

# 日本語フォントの設定
try:
  import japanize_matplotlib
except ImportError:
  plt.rcParams['font.family'] = 'sans-serif'

# 1. 抽出したい時間区間[μs]の定義
INTERVALS = {
    '1': (74, 114),
    '2': (149, 189),
    '3': (300, 340),
    '4': (375, 415),
    '5': (564, 604),
    '6': (639, 679),
}

# ハイパスフィルタの設定
CUTOFF_HZ = 200e3
FILTER_ORDER = 4

# 対象ファイルの取得
target_pattern = f'source/{project_name}/**/*.csv'
csv_files = sorted(glob.glob(target_pattern, recursive=True))

if not csv_files:
  print('対象のCSVファイルが見つかりませんでした。')
else:
  num_files = len(csv_files)
  summary_data = []  # 数値データを集積するリスト

  # フィルタ前用とフィルタ後用の2セットのキャンバスを作成
  fig_raw, axes_raw = plt.subplots(
      nrows=num_files, ncols=1, figsize=(10, 2.5 * num_files), sharex=True
  )
  fig_hpf, axes_hpf = plt.subplots(
      nrows=num_files, ncols=1, figsize=(10, 2.5 * num_files), sharex=True
  )

  if num_files == 1:
    axes_raw = [axes_raw]
    axes_hpf = [axes_hpf]

  x_col = '時間[μs]'
  y_col = ' 加算平均値[V]'

  for ax_r, ax_h, file_path in zip(axes_raw, axes_hpf, csv_files):
    try:
      df = pd.read_csv(file_path, encoding='ms932')
    except (UnicodeDecodeError, pd.errors.ParserError):
      df = pd.read_csv(file_path, encoding='cp932')

    if x_col in df.columns and y_col in df.columns:
      folder_and_file = '/'.join(Path(file_path).parts[-2:])
      time_us = df[x_col].values
      raw_mv = df[y_col].values * 1000  # V -> mV 変換

      # --- フィルタ処理 ---
      dt_us = np.mean(np.diff(time_us))
      fs = 1e6 / dt_us
      nyq = 0.5 * fs

      if CUTOFF_HZ < nyq:
        normal_cutoff = CUTOFF_HZ / nyq
        b, a = butter(FILTER_ORDER, normal_cutoff, btype='highpass')
        hpf_mv = filtfilt(b, a, raw_mv)
      else:
        hpf_mv = np.full_like(raw_mv, np.nan)

      # --- 1. 基本波形のプロット ---
      ax_r.plot(time_us, raw_mv, color='tab:blue', linewidth=1.2)
      ax_h.plot(time_us, hpf_mv, color='tab:orange', linewidth=1.2)

      # --- 2. 各区間の数値計算 ＆ ハイライト表示 ---
      for sec_num, (t_start, t_end) in INTERVALS.items():
        # 両方のグラフに黄色半透明でハイライト背景を付与
        for ax in [ax_r, ax_h]:
          ax.axvspan(t_start, t_end, color='gold', alpha=0.25)

        # 指定時間区間のデータを切り出し
        mask = (time_us >= t_start) & (time_us <= t_end)

        # フィルタ適用前 (Raw) の計算
        r_sub = raw_mv[mask]
        r_max = np.max(r_sub) if len(r_sub) > 0 else np.nan
        r_min = np.min(r_sub) if len(r_sub) > 0 else np.nan
        r_p2p = r_max - r_min if len(r_sub) > 0 else np.nan

        # フィルタ適用後 (HPF) の計算
        h_sub = hpf_mv[mask]
        if len(h_sub) > 0 and not np.isnan(h_sub).all():
          h_max = np.max(h_sub)
          h_min = np.min(h_sub)
          h_p2p = h_max - h_min
        else:
          h_max = h_min = h_p2p = np.nan

        # リストに数値を保持
        summary_data.append({
            'ファイル名': folder_and_file,
            '区間番号': sec_num,
            '時間帯[μs]': f'{t_start}-{t_end}',
            'Raw_最大値[mV]': r_max,
            'Raw_最小値[mV]': r_min,
            'Raw_P2P[mV]': r_p2p,
            'HPF_最大値[mV]': h_max,
            'HPF_最小値[mV]': h_min,
            'HPF_P2P[mV]': h_p2p,
        })

      # 各サブプロットの軸設定
      for ax, title_suffix in zip(
          [ax_r, ax_h], ['(Raw)', '(HPF 200kHz)']
      ):
        ax.set_title(
            f'{folder_and_file} {title_suffix}',
            fontsize=10,
            loc='left',
            fontweight='bold',
        )
        ax.set_ylabel('加算平均値[mV]', fontsize=9)
        ax.set_xlim(0, 700)
        ax.set_ylim(-5, 5)
        ax.grid(True, linestyle='--', alpha=0.6)

    else:
      for ax in [ax_r, ax_h]:
        ax.text(
            0.5,
            0.5,
            f'指定カラムが見つかりません:\n{file_path}',
            ha='center',
            va='center',
            transform=ax.transAxes,
        )

  # X軸ラベルの設定
  axes_raw[-1].set_xlabel(x_col, fontsize=10)
  axes_hpf[-1].set_xlabel(x_col, fontsize=10)

  # --- 3. 出力処理 ---
  output_dir = Path(f'./results/{project_name}')
  output_dir.mkdir(parents=True, exist_ok=True)

  # フィルタ前画像の保存
  fig_raw.tight_layout()
  fig_raw.savefig(
      output_dir / 'comparison_raw_highlighted.png',
      dpi=300,
      bbox_inches='tight',
  )

  # フィルタ後画像の保存
  fig_hpf.tight_layout()
  fig_hpf.savefig(
      output_dir / 'comparison_hpf_highlighted.png',
      dpi=300,
      bbox_inches='tight',
  )

  plt.show()  # フィルタ後画像を画面に表示

  # 4. 数値結果のDataFrame化とCSV出力
  df_summary = pd.DataFrame(summary_data)
  csv_path = output_dir / 'p2p_summary.csv'
  # 文字化け(Excel対応)防止のため utf-8-sig で保存
  df_summary.to_csv(csv_path, index=False, encoding='utf-8-sig')

  print('=== 処理完了 ===')
  print(f'1. 数値一覧データ : "{csv_path}"')
  print(
      f'2. フィルタ前画像 : "{output_dir / "comparison_raw_highlighted.png"}"'
  )
  print(
      f'3. フィルタ後画像 : "{output_dir / "comparison_hpf_highlighted.png"}"'
  )

# セル上で数値のテーブルを表示
df_summary

## 歪-磁場曲線の出力

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

# 1. FolderMaster.csv の読み込み
df_master = pd.read_csv(master_csv_path)

# WithCancel の文字列/数値表記をブール値 (True/False) に標準化
if df_master['WithCancel'].dtype == object:
  df_master['WithCancel'] = (
      df_master['WithCancel']
      .astype(str)
      .str.strip()
      .str.lower()
      .map({'true': True, 'false': False, '1': True, '0': False})
  )

# 2. P2Pデータ (df_summary) と FolderMaster を結合
df_summary['FolderName'] = df_summary['ファイル名'].apply(
    lambda x: x.split('/')[0]
)
df_merged = pd.merge(df_summary, df_master, on='FolderName', how='left')
df_merged = df_merged.sort_values(by='MagneticField')

# 出力フォルダの用意
output_dir = Path(f'./results/{project_name}')
output_dir.mkdir(parents=True, exist_ok=True)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

print('データの準備が完了しました。')

In [ ]:
import numpy as np

# --- WithCancel = False の描画セル ---
df_false = df_merged[df_merged['WithCancel'] == False]

# 近似曲線の滑らかさ（多項式の次数: 2=2次曲線/パラボラ, 3=3次曲線）
DEGREE = 3

if df_false.empty:
  print('WithCancel = False のデータが存在しません。')
else:
  fig, ax = plt.subplots(figsize=(10, 6))
  intervals = sorted(df_false['区間番号'].unique())

  for idx, sec_num in enumerate(intervals):
    df_sec = df_false[df_false['区間番号'] == sec_num].dropna(
        subset=['MagneticField', 'HPF_P2P[mV]']
    )
    time_range = df_sec['時間帯[μs]'].iloc[0] if not df_sec.empty else ''
    color = colors[idx % len(colors)]

    x = df_sec['MagneticField'].values
    y_hpf = df_sec['HPF_P2P[mV]'].values
    y_raw = df_sec['Raw_P2P[mV]'].values

    # 1. 測定点（マーカー）の描画
    ax.scatter(
        x,
        y_hpf,
        color=color,
        s=40,
        zorder=3,
        label=f'区間{sec_num} ({time_range}μs) [HPF]',
    )
    ax.scatter(x, y_raw, color=color, marker='s', s=30, alpha=0.35, zorder=2)

    # 2. 多項式近似による滑らかな曲線の算出・描画
    if len(x) > DEGREE:
      # 滑らかなX軸データを生成 (200分割)
      x_smooth = np.linspace(x.min(), x.max(), 200)

      # HPFの近似曲線
      coeffs_hpf = np.polyfit(x, y_hpf, DEGREE)
      poly_hpf = np.poly1d(coeffs_hpf)
      ax.plot(
          x_smooth,
          poly_hpf(x_smooth),
          color=color,
          linestyle='-',
          linewidth=1.8,
      )

      # Rawの近似曲線（破線）
      coeffs_raw = np.polyfit(x, y_raw, DEGREE)
      poly_raw = np.poly1d(coeffs_raw)
      ax.plot(
          x_smooth,
          poly_raw(x_smooth),
          color=color,
          linestyle='--',
          linewidth=1.0,
          alpha=0.35,
      )
    else:
      # データ点が少ない場合はそのまま直線接続
      ax.plot(x, y_hpf, color=color, linestyle='-', linewidth=1.8)
      ax.plot(x, y_raw, color=color, linestyle='--', linewidth=1.0, alpha=0.35)

  ax.set_title(
      'P2P vs MagneticField (WithCancel = False) [Polynomial Fit]',
      fontsize=12,
      fontweight='bold',
  )
  ax.set_xlabel('MagneticField [T]', fontsize=11)
  ax.set_ylabel('Peak-to-Peak (P2P) [mV]', fontsize=11)
  ax.grid(True, linestyle='--', alpha=0.6)
  ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8.5)
  ax.set_xlim(0, 2.5)
  ax.set_ylim(0, 20)

  plt.tight_layout()
  save_path_false = (
      output_dir / 'p2p_vs_magneticfield_WithCancel_False_smooth.png'
  )
  plt.savefig(save_path_false, dpi=300, bbox_inches='tight')
  plt.show()

  print(f'Falseグラフを出力しました: "{save_path_false}"')

In [ ]:
import numpy as np

# --- WithCancel = True の描画セル ---
df_true = df_merged[df_merged['WithCancel'] == True]

# 近似曲線の滑らかさ（多項式の次数: 2=2次曲線, 3=3次曲線）
DEGREE = 3

if df_true.empty:
  print('WithCancel = True のデータが存在しません。')
else:
  fig, ax = plt.subplots(figsize=(10, 6))
  intervals = sorted(df_true['区間番号'].unique())

  for idx, sec_num in enumerate(intervals):
    df_sec = df_true[df_true['区間番号'] == sec_num].dropna(
        subset=['MagneticField', 'HPF_P2P[mV]']
    )
    time_range = df_sec['時間帯[μs]'].iloc[0] if not df_sec.empty else ''
    color = colors[idx % len(colors)]

    x = df_sec['MagneticField'].values
    y_hpf = df_sec['HPF_P2P[mV]'].values
    y_raw = df_sec['Raw_P2P[mV]'].values

    # 1. 測定点（マーカー）の描画
    ax.scatter(
        x,
        y_hpf,
        color=color,
        s=40,
        zorder=3,
        label=f'区間{sec_num} ({time_range}μs) [HPF]',
    )
    ax.scatter(x, y_raw, color=color, marker='s', s=30, alpha=0.35, zorder=2)

    # 2. 多項式近似による滑らかな曲線の算出・描画
    if len(x) > DEGREE:
      # 滑らかなX軸データを生成 (200分割)
      x_smooth = np.linspace(x.min(), x.max(), 200)

      # HPFの近似曲線
      coeffs_hpf = np.polyfit(x, y_hpf, DEGREE)
      poly_hpf = np.poly1d(coeffs_hpf)
      ax.plot(
          x_smooth,
          poly_hpf(x_smooth),
          color=color,
          linestyle='-',
          linewidth=1.8,
      )

      # Rawの近似曲線（破線）
      coeffs_raw = np.polyfit(x, y_raw, DEGREE)
      poly_raw = np.poly1d(coeffs_raw)
      ax.plot(
          x_smooth,
          poly_raw(x_smooth),
          color=color,
          linestyle='--',
          linewidth=1.0,
          alpha=0.35,
      )
    else:
      # データ点が少ない場合はそのまま直線接続
      ax.plot(x, y_hpf, color=color, linestyle='-', linewidth=1.8)
      ax.plot(x, y_raw, color=color, linestyle='--', linewidth=1.0, alpha=0.35)

  ax.set_title(
      'P2P vs MagneticField (WithCancel = True) [Polynomial Fit]',
      fontsize=12,
      fontweight='bold',
  )
  ax.set_xlabel('MagneticField [T]', fontsize=11)
  ax.set_ylabel('Peak-to-Peak (P2P) [mV]', fontsize=11)
  ax.grid(True, linestyle='--', alpha=0.6)
  ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8.5)
  ax.set_xlim(0, 2.5)
  ax.set_ylim(0, 20)

  plt.tight_layout()
  save_path_true = (
      output_dir / 'p2p_vs_magneticfield_WithCancel_True_smooth.png'
  )
  plt.savefig(save_path_true, dpi=300, bbox_inches='tight')
  plt.show()

  print(f'Trueグラフを出力しました: "{save_path_true}"')